# Fabric Connection Audit

Inventory the data connections the calling identity can see, flag **stale**
ones, **rename** them to a naming convention, and optionally **take
ownership** of the ones you already have the right to manage.

**Usage:**
1. Review the **Configuration** cell. Every write is gated: `DRY_RUN = True`
   previews renames and ownership grants without touching anything.
2. Run the audit to see each connection, its staleness, and whether this
   identity can manage it.
3. Edit `propose_name()` to encode your convention, preview the old → new
   mapping, then set `DRY_RUN = False` and `RENAME = True` to apply.

**Permission boundary.** The Connections API has no self-escalation path.
You can only rename or reassign a connection where the caller already holds
**UserWithReshare / Owner**, or is **Admin on the bound gateway**. A
connection you have no role on is invisible to `GET /v1/connections` in the
first place; one you hold only plain `User` on shows up but reports
`manageable = False`. "Taking ownership" here means granting `Owner` on
connections you can *already* manage - it cannot pull in orphaned
connections nobody on your identity can reach.

**Identity.** Runs under the notebook runtime token - the running user, or a
service principal when scheduled. That identity is the one whose rights bound
everything below.

## Configuration

Both write gates ship closed. `DRY_RUN = True` is the master switch - while
it is set, `RENAME` and `TAKE_OWNERSHIP` only preview. Set `DRY_RUN = False`
**and** the specific action flag to apply that action.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────
# Master safety gate. While True nothing is written - renames and ownership
# grants are previewed only. Flip to False to let the action flags below fire.
DRY_RUN = True

# Staleness threshold. A connection is flagged stale when it has not been bound
# to an item, nor had its credential used, within this many days - or when it
# carries no recency data at all.
STALE_AFTER_DAYS = 90

# Rename connections to match propose_name(). Requires DRY_RUN = False.
RENAME = False

# Only rename connections currently flagged stale. False renames every
# manageable connection whose name differs from propose_name().
RENAME_STALE_ONLY = False

# Grant Owner on targeted connections to the principal below. Requires
# DRY_RUN = False. Bounded by the caller's own reshare/owner/gateway-admin
# rights - there is no self-escalation.
TAKE_OWNERSHIP = False
OWNER_PRINCIPAL_ID = "<TargetPrincipalObjectId>"   # Entra object ID, not a UPN
OWNER_PRINCIPAL_TYPE = "User"                       # User | ServicePrincipal | Group

# Only take ownership of connections currently flagged stale. False targets
# every manageable connection.
OWN_STALE_ONLY = True

## Authentication & request layer

Acquires a Fabric token from the notebook runtime and defines the shared
request layer: `fabric_request` (429 / transient-5xx back-off honoring
`Retry-After`, plus a one-shot token refresh on 401) and `get_paginated`
(follows `continuationUri` to the end of a list endpoint).

In [ ]:
import re
import time
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional

import requests

# ── Authenticate via Fabric runtime ───────────────────────────────────────
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
FABRIC_TOKEN_SCOPE = "https://api.fabric.microsoft.com/.default"

MAX_RETRIES = 5            # per request, for 429 and transient 5xx
DEFAULT_RETRY_AFTER = 5    # seconds, when Retry-After is absent or unparseable

TOKEN = ""
HEADERS: Dict[str, str] = {}


def refresh_token() -> None:
    """Acquire a Fabric API token and rebuild the shared request headers.

    :returns: None. Mutates the module-level TOKEN and HEADERS.
    """
    global TOKEN, HEADERS
    TOKEN = notebookutils.credentials.getToken(FABRIC_TOKEN_SCOPE)
    HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


refresh_token()


def parse_retry_after(response: requests.Response, default: int = DEFAULT_RETRY_AFTER) -> int:
    """Read the Retry-After header as a delay in seconds.

    RFC 9110 permits an HTTP-date instead of a delay-seconds integer, so a bare
    int() would raise on a legal response. Fall back to the default instead.

    :param response: Response whose headers to inspect.
    :param default: Seconds to use when the header is missing or non-integer.
    :returns: A positive number of seconds to wait.
    """
    raw = response.headers.get("Retry-After")
    if raw is None:
        return default
    try:
        return max(1, int(raw))
    except (TypeError, ValueError):
        return default


def fabric_request(method: str, url: str, json_body: Optional[Dict] = None,
                   max_retries: int = MAX_RETRIES) -> requests.Response:
    """Issue a Fabric REST call with throttling back-off and one token refresh.

    Retries 429 and transient 5xx responses honoring Retry-After, and refetches
    the token once on a 401. Non-retriable responses are returned as-is for the
    caller to interpret.

    :param method: HTTP verb, e.g. "GET", "POST", "PATCH".
    :param url: Absolute request URL.
    :param json_body: Optional JSON body to send.
    :param max_retries: Maximum retry attempts for throttled / transient failures.
    :returns: The final requests.Response, successful or not.
    """
    response = None
    refreshed = False

    for attempt in range(max_retries + 1):
        response = requests.request(method, url, headers=HEADERS, json=json_body)

        if response.status_code == 401 and not refreshed:
            print("  401 Unauthorized - refreshing token and retrying once...")
            refresh_token()
            refreshed = True
            continue

        if response.status_code == 429 or response.status_code in (502, 503, 504):
            if attempt == max_retries:
                break
            wait = parse_retry_after(response)
            print(
                f"  {response.status_code} from {url} - retrying in {wait}s "
                f"(attempt {attempt + 1}/{max_retries})"
            )
            time.sleep(wait)
            continue

        break

    return response


def get_paginated(url: str, what: str) -> List[Dict]:
    """Follow continuationUri to the end of a Fabric list endpoint.

    :param url: First-page URL.
    :param what: Noun used in the error message, e.g. "connections".
    :returns: Concatenated `value` arrays from every page.
    :raises RuntimeError: If any page returns a non-200 status.
    """
    collected: List[Dict] = []

    while url:
        response = fabric_request("GET", url)
        if response.status_code != 200:
            raise RuntimeError(
                f"Failed to list {what}: {response.status_code} {response.text}"
            )
        data = response.json()
        collected.extend(data.get("value", []))
        url = data.get("continuationUri")

    return collected


print("Authenticated. Request layer ready.")

## Connection helpers

Staleness, manageability, rename, and ownership. Two API facts are encoded
here:

- **Rename is not supported for every connectivity type.** `PersonalCloud`
  and `OnPremisesGatewayPersonal` update requests carry no `displayName`
  field, so those are skipped rather than failing mid-batch.
- **Manageability is probed by reading role assignments.** A `200` implies
  Owner / gateway-admin; a `403` is the API declining, and marks the
  connection `manageable = False`.

In [ ]:
# ── Connectivity types whose update request accepts a displayName ───────────
# PersonalCloud and OnPremisesGatewayPersonal are deliberately excluded - their
# update contracts have no displayName field, so a rename there is a no-op at
# best and a 400 at worst.
RENAMEABLE_TYPES = {
    "ShareableCloud",
    "OnPremisesGateway",
    "VirtualNetworkGateway",
    "StreamingVirtualNetworkGateway",
}


def parse_ts(stamp: str) -> datetime:
    """Parse a Fabric UTC timestamp (YYYY-MM-DDTHH:mm:ssZ) to an aware datetime.

    :param stamp: Timestamp string from a connectionRecency field.
    :returns: Timezone-aware UTC datetime.
    """
    return datetime.strptime(stamp, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)


def last_activity(conn: Dict) -> Optional[datetime]:
    """Return the most recent bind-or-credential-use time for a connection.

    :param conn: A connection object from the list endpoint.
    :returns: Latest recency datetime, or None when no recency data is present.
    """
    recency = conn.get("connectionRecency") or {}
    stamps = [
        recency.get("lastBoundDateTime"),
        recency.get("lastCredentialUsedDateTime"),
    ]
    parsed = [parse_ts(s) for s in stamps if s]
    return max(parsed) if parsed else None


def is_stale(conn: Dict, cutoff: datetime) -> bool:
    """Decide whether a connection is stale relative to a cutoff.

    A connection with no recency data is treated as stale - it has no evidence
    of recent use.

    :param conn: A connection object from the list endpoint.
    :param cutoff: Connections last active before this are stale.
    :returns: True when stale.
    """
    activity = last_activity(conn)
    return activity is None or activity < cutoff


def is_manageable(conn: Dict) -> bool:
    """Probe whether the caller can manage a connection (Owner / gateway-admin).

    :param conn: A connection object.
    :returns: True when the role-assignments read returns 200.
    """
    url = f"{FABRIC_API_BASE}/connections/{conn['id']}/roleAssignments"
    return fabric_request("GET", url).status_code == 200


def rename_connection(conn: Dict, new_name: str) -> Optional[Dict]:
    """PATCH a connection's display name, skipping non-renameable types.

    :param conn: A connection object (must carry id and connectivityType).
    :param new_name: Desired display name (max 200 chars).
    :returns: The updated connection object, or None if skipped or failed.
    """
    ctype = conn.get("connectivityType")
    name = conn.get("displayName")
    if ctype not in RENAMEABLE_TYPES:
        print(f"  SKIP  {name}: {ctype} connections cannot be renamed via API")
        return None

    body = {"connectivityType": ctype, "displayName": new_name}
    response = fabric_request(
        "PATCH", f"{FABRIC_API_BASE}/connections/{conn['id']}", json_body=body
    )
    if response.status_code != 200:
        print(f"  FAIL  {name} -> {new_name}: {response.status_code} {response.text}")
        return None

    print(f"  OK    {name} -> {new_name}")
    return response.json()


def grant_owner(conn: Dict, principal_id: str, principal_type: str = "User") -> Optional[Dict]:
    """Grant the Owner role on a connection to a principal.

    :param conn: A connection object.
    :param principal_id: Entra object ID of the principal to grant.
    :param principal_type: User | ServicePrincipal | Group | etc.
    :returns: The created role-assignment object, or None on failure.
    """
    url = f"{FABRIC_API_BASE}/connections/{conn['id']}/roleAssignments"
    body = {"principal": {"id": principal_id, "type": principal_type}, "role": "Owner"}
    response = fabric_request("POST", url, json_body=body)
    if response.status_code not in (200, 201):
        print(
            f"  FAIL  owner grant on {conn.get('displayName')}: "
            f"{response.status_code} {response.text}"
        )
        return None

    print(f"  OK    Owner -> {principal_type} {principal_id} on {conn.get('displayName')}")
    return response.json()

## Audit

Lists every visible connection and, for each, records its source, credential
type, last activity, staleness, and whether this identity can manage it. The
result is held in `AUDIT` for the rename and ownership cells below.

In [ ]:
# ── Enumerate + classify ──────────────────────────────────────────────
cutoff = datetime.now(timezone.utc) - timedelta(days=STALE_AFTER_DAYS)
connections = get_paginated(f"{FABRIC_API_BASE}/connections", "connections")

AUDIT: List[Dict] = []
for conn in connections:
    activity = last_activity(conn)
    AUDIT.append({
        "conn": conn,
        "id": conn["id"],
        "name": conn.get("displayName", "(no name)"),
        "connectivityType": conn.get("connectivityType"),
        "source": conn.get("connectionDetails", {}).get("path"),
        "credentialType": conn.get("credentialDetails", {}).get("credentialType"),
        "lastActivity": activity.strftime("%Y-%m-%d") if activity else "never/unknown",
        "stale": is_stale(conn, cutoff),
        "manageable": is_manageable(conn),
    })

stale_count = sum(1 for r in AUDIT if r["stale"])
manageable_count = sum(1 for r in AUDIT if r["manageable"])
print(
    f"{len(AUDIT)} visible connection(s): "
    f"{stale_count} stale (> {STALE_AFTER_DAYS}d), {manageable_count} manageable\n"
)

for r in sorted(AUDIT, key=lambda x: (not x["stale"], x["name"].lower())):
    flags = "STALE" if r["stale"] else "     "
    lock = "      " if r["manageable"] else "NoMgmt"
    print(f"[{flags}][{lock}] {r['name']}  ({r['connectivityType']})")
    print(f"            id:        {r['id']}")
    print(f"            source:    {r['source']}")
    print(f"            cred:      {r['credentialType']}   last active: {r['lastActivity']}")

## Rename to a naming convention

Edit `propose_name()` to encode your convention - it returns the desired name
for a connection, or `None` to leave it alone. The shipped example builds
`<sourcetype>-<slug-of-path>`; replace it with yours. The cell previews the
`old → new` mapping for every manageable, renameable connection whose name
differs, and only applies when `DRY_RUN = False` and `RENAME = True`.

In [ ]:
def propose_name(conn: Dict) -> Optional[str]:
    """Return the desired display name for a connection, or None to skip it.

    EDIT ME to match your convention. The example derives a name from the
    connection's source: an Azure SQL connection to
    'contoso.database.windows.net;sales' becomes 'sql-contoso-database-
    windows-net-sales'.

    :param conn: A connection object.
    :returns: Proposed display name (<= 200 chars), or None to leave as-is.
    """
    details = conn.get("connectionDetails", {})
    source_type = (details.get("type") or "conn").lower()
    path = details.get("path") or ""
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", path).strip("-").lower()
    if not slug:
        return None
    return f"{source_type}-{slug}"[:200]


# ── Build the rename plan ───────────────────────────────────────────
plan = []
for r in AUDIT:
    if not r["manageable"] or r["connectivityType"] not in RENAMEABLE_TYPES:
        continue
    if RENAME_STALE_ONLY and not r["stale"]:
        continue
    proposed = propose_name(r["conn"])
    if proposed and proposed != r["name"]:
        plan.append((r, proposed))

print(f"{len(plan)} connection(s) would be renamed:\n")
for r, proposed in plan:
    print(f"  {r['name']}")
    print(f"    → {proposed}")

# ── Apply ───────────────────────────────────────────────────────
if plan and not DRY_RUN and RENAME:
    print("\nApplying renames...")
    for r, proposed in plan:
        rename_connection(r["conn"], proposed)
else:
    print("\nPreview only (set DRY_RUN = False and RENAME = True to apply).")

## Take ownership (optional)

Grants `Owner` on targeted connections to `OWNER_PRINCIPAL_ID`. This only
works where the caller already holds UserWithReshare/Owner or is gateway
Admin - i.e. the `manageable = True` set from the audit. Guarded behind
`DRY_RUN = False` and `TAKE_OWNERSHIP = True`.

In [ ]:
# ── Build the ownership plan ────────────────────────────────────────
targets = [
    r for r in AUDIT
    if r["manageable"] and (r["stale"] or not OWN_STALE_ONLY)
]

print(f"{len(targets)} connection(s) would be granted to "
      f"{OWNER_PRINCIPAL_TYPE} {OWNER_PRINCIPAL_ID}:\n")
for r in targets:
    print(f"  {r['name']}  ({r['connectivityType']})")

# ── Apply ───────────────────────────────────────────────────────
if targets and not DRY_RUN and TAKE_OWNERSHIP:
    if OWNER_PRINCIPAL_ID.startswith("<"):
        raise ValueError("Set OWNER_PRINCIPAL_ID to a real Entra object ID first.")
    print("\nGranting ownership...")
    for r in targets:
        grant_owner(r["conn"], OWNER_PRINCIPAL_ID, OWNER_PRINCIPAL_TYPE)
else:
    print("\nPreview only (set DRY_RUN = False and TAKE_OWNERSHIP = True to apply).")